# T5-base Fine-Tuning on Spider
- Upload to Drive under `MyDrive/text-to-sql/spider_data/`:
  - `train_spider.json`, `train_others.json`, `dev.json`, `tables.json`

## 1. Setup

In [1]:
!pip install -q "transformers>=4.40" "datasets>=2.14" "accelerate>=0.30"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch
import random
import numpy as np
from pathlib import Path

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DRIVE_ROOT = Path('/content/drive/MyDrive/text-to-sql')
SPIDER_DIR = DRIVE_ROOT / 'spider_data'
CHECKPOINT_DIR = DRIVE_ROOT / 't5_base_spider'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

for f in ['train_spider.json', 'train_others.json', 'dev.json', 'tables.json']:
    assert (SPIDER_DIR / f).exists(), f'Missing: {SPIDER_DIR / f}'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU. Change Runtime → GPU.')

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


2. Load Spider & build schema strings

In [5]:
import json

def load_json(path):
    with open(path) as f:
        return json.load(f)

train = load_json(SPIDER_DIR / 'train_spider.json')
train_others = load_json(SPIDER_DIR / 'train_others.json')
validation = load_json(SPIDER_DIR / 'dev.json')
tables = load_json(SPIDER_DIR / 'tables.json')

print(f'train_spider: {len(train)}')
print(f'train_others: {len(train_others)}')
print(f'validation: {len(validation)}')
print(f'schemas: {len(tables)}')

train_spider: 7000
train_others: 1659
validation: 1034
schemas: 166


In [6]:
def build_schema_string(db):
    table_names = db['table_names_original']
    columns = db['column_names_original']
    column_types = db['column_types']
    foreign_keys = db.get('foreign_keys', [])

    tables_to_cols = {i: [] for i in range(len(table_names))}
    col_idx_to_ref = {}
    for col_idx, (tbl_idx, col_name) in enumerate(columns):
        if tbl_idx == -1:
            continue
        col_type = column_types[col_idx]
        tables_to_cols[tbl_idx].append(f'{col_name}:{col_type}')
        col_idx_to_ref[col_idx] = f'{table_names[tbl_idx]}.{col_name}'

    table_parts = []
    for i, tbl in enumerate(table_names):
        cols = ', '.join(tables_to_cols[i])
        table_parts.append(f'{tbl}({cols})')

    fk_parts = []
    for a, b in foreign_keys:
        if a in col_idx_to_ref and b in col_idx_to_ref:
            fk_parts.append(f'{col_idx_to_ref[a]} = {col_idx_to_ref[b]}')

    schema = ' | '.join(table_parts)
    if fk_parts:
        schema += ' | foreign_keys: ' + ', '.join(fk_parts)
    return schema

schema_by_db = {t['db_id']: build_schema_string(t) for t in tables}
print('Sample schema (concert_singer):')
print(schema_by_db['concert_singer'])

Sample schema (concert_singer):
stadium(Stadium_ID:number, Location:text, Name:text, Capacity:number, Highest:number, Lowest:number, Average:number) | singer(Singer_ID:number, Name:text, Country:text, Song_Name:text, Song_release_year:text, Age:number, Is_male:others) | concert(concert_ID:number, concert_Name:text, Theme:text, Stadium_ID:text, Year:text) | singer_in_concert(concert_ID:number, Singer_ID:text) | foreign_keys: concert.Stadium_ID = stadium.Stadium_ID, singer_in_concert.Singer_ID = singer.Singer_ID, singer_in_concert.concert_ID = concert.concert_ID


In [7]:
def make_input(ex):
    return f"translate English to SQL | {ex['db_id']} | {ex['question']} | schema: {schema_by_db[ex['db_id']]}"

def clean_target(sql):
    return ' '.join(sql.split())

def to_t5_pairs(examples):
    return [
        {'input': make_input(ex), 'target': clean_target(ex['query'])}
        for ex in examples
    ]

train_pairs = to_t5_pairs(train) + to_t5_pairs(train_others)
val_pairs = to_t5_pairs(validation)

print(f'train pairs: {len(train_pairs)}')
print(f'val pairs: {len(val_pairs)}')
print()
print('sample input:')
print(train_pairs[0]['input'])
print()
print('sample target:', train_pairs[0]['target'])

train pairs: 8659
val pairs: 1034

sample input:
translate English to SQL | department_management | How many heads of the departments are older than 56 ? | schema: department(Department_ID:number, Name:text, Creation:text, Ranking:number, Budget_in_Billions:number, Num_Employees:number) | head(head_ID:number, name:text, born_state:text, age:number) | management(department_ID:number, head_ID:number, temporary_acting:text) | foreign_keys: management.head_ID = head.head_ID, management.department_ID = department.Department_ID

sample target: SELECT count(*) FROM head WHERE age > 56


## 3. Tokenize

In [8]:
from transformers import AutoTokenizer

MODEL_NAME = 't5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN = 768
MAX_TARGET_LEN = 256

input_lens = [len(tokenizer.encode(p['input'])) for p in train_pairs[:500]]
target_lens = [len(tokenizer.encode(p['target'])) for p in train_pairs[:500]]
print(f'Input tokens — p50: {np.percentile(input_lens, 50):.0f}, p95: {np.percentile(input_lens, 95):.0f}, max: {max(input_lens)}')
print(f'Target tokens — p50: {np.percentile(target_lens, 50):.0f}, p95: {np.percentile(target_lens, 95):.0f}, max: {max(target_lens)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input tokens — p50: 165, p95: 593, max: 604
Target tokens — p50: 28, p95: 74, max: 151


In [9]:
from datasets import Dataset

def tokenize_fn(batch):
    model_inputs = tokenizer(
        batch['input'], max_length=MAX_INPUT_LEN, truncation=True
    )
    labels = tokenizer(
        text_target=batch['target'], max_length=MAX_TARGET_LEN, truncation=True
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_list(train_pairs).map(
    tokenize_fn, batched=True, remove_columns=['input', 'target']
)
val_ds = Dataset.from_list(val_pairs).map(
    tokenize_fn, batched=True, remove_columns=['input', 'target']
)

print(train_ds)
print(val_ds)

Map:   0%|          | 0/8659 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8659
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1034
})


## 4. Train
- 8 epochs

In [11]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=False,
    fp16=True,
    report_to='none',
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
existing_ckpts = sorted(CHECKPOINT_DIR.glob('checkpoint-*'))
resume = len(existing_ckpts) > 0

if resume:
    print(f'Resuming from {len(existing_ckpts)} existing checkpoint(s):')
    for c in existing_ckpts:
        print(f'  {c.name}')
else:
    print('Starting fresh training run')

trainer.train(resume_from_checkpoint=resume)

Starting fresh training run


Epoch,Training Loss,Validation Loss
1,0.403832,0.466465
2,0.217479,0.419028
3,0.158139,0.415075
4,0.146118,0.383961
5,0.137131,0.392856
6,0.141710,0.398139
7,0.140087,0.398106
8,0.134873,0.398098


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4336, training_loss=0.2956943717609912, metrics={'train_runtime': 973.5635, 'train_samples_per_second': 71.153, 'train_steps_per_second': 4.454, 'total_flos': 6.117503430868992e+16, 'train_loss': 0.2956943717609912, 'epoch': 8.0})

In [13]:
FINAL_DIR = CHECKPOINT_DIR / 'final'
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
print(f'Best model (by eval loss) saved to {FINAL_DIR}')
print()
print('Eval loss per epoch:')
for log in trainer.state.log_history:
    if 'eval_loss' in log:
        print(f"  epoch {log['epoch']:.1f}: eval_loss = {log['eval_loss']:.4f}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model (by eval loss) saved to /content/drive/MyDrive/text-to-sql/t5_base_spider/final

Eval loss per epoch:
  epoch 1.0: eval_loss = 0.4665
  epoch 2.0: eval_loss = 0.4190
  epoch 3.0: eval_loss = 0.4151
  epoch 4.0: eval_loss = 0.3840
  epoch 5.0: eval_loss = 0.3929
  epoch 6.0: eval_loss = 0.3981
  epoch 7.0: eval_loss = 0.3981
  epoch 8.0: eval_loss = 0.3981



quick peek at some predictions before full scale inference (looks good)


In [14]:
model.eval()
for ex in val_pairs[:5]:
    inputs = tokenizer(
        ex['input'], return_tensors='pt', truncation=True, max_length=MAX_INPUT_LEN
    ).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_TARGET_LEN, num_beams=4, early_stopping=True
        )
    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    print('GOLD:', ex['target'])
    print('PRED:', pred)
    print()

GOLD: SELECT count(*) FROM singer
PRED: SELECT count(*) FROM singer

GOLD: SELECT count(*) FROM singer
PRED: SELECT count(*) FROM singer

GOLD: SELECT name , country , age FROM singer ORDER BY age DESC
PRED: SELECT name , country , age FROM singer ORDER BY age

GOLD: SELECT name , country , age FROM singer ORDER BY age DESC
PRED: SELECT name , country , age FROM singer ORDER BY age DESC

GOLD: SELECT avg(age) , min(age) , max(age) FROM singer WHERE country = 'France'
PRED: SELECT avg(age) , min(age) , max(age) FROM singer WHERE country = 'France'



## 6. Cleanup intermediate checkpoints (optional)

After `final/` is saved, you can drop the intermediate `checkpoint-XXXX/` folders from Drive to free space.

In [15]:
import shutil

DELETE_CHECKPOINTS = False

if DELETE_CHECKPOINTS:
    for ckpt in CHECKPOINT_DIR.glob('checkpoint-*'):
        print(f'Deleting {ckpt}')
        shutil.rmtree(ckpt)
    print('Done')
else:
    print('Set DELETE_CHECKPOINTS = True and re-run this cell to clean up')

Set DELETE_CHECKPOINTS = True and re-run this cell to clean up
